# C16 — Manifestes de CaFE et d'OpenSLR 57 (P7, P9, P10)

Ce notebook télécharge les deux jeux hors domaine, construit leur manifeste avec le code du dépôt
(`echolens.core.data.cafe` et `echolens.core.data.openslr57`), puis vérifie les effectifs
attendus. Il n'écrit que deux CSV de métadonnées, `cafe.csv` et `openslr57.csv`, sans aucun
audio (I8). **L'option Internet du notebook doit être activée.**

## Démarrage

On clone le dépôt (branche `BRANCH`) et on l'installe en mode éditable (I9), puis on ajoute
`src/` au `sys.path` : l'installation éditable n'est visible qu'après un redémarrage du noyau.

In [ ]:
import sys

BRANCH = "feat/c16-manifestes-hors-domaine"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## a. Téléchargement

Sources officielles : l'archive CaFE 48 kHz sur Zenodo (records/1478765, environ 1 Go) et
l'archive d'OpenSLR 57 (environ 1,8 Go). Les archives sont supprimées dès qu'elles sont
décompressées, pour ne pas remplir le disque de Kaggle. Compter une dizaine de minutes.

In [ ]:
import tarfile
import urllib.request
import zipfile
from pathlib import Path

DOWNLOAD_DIR = Path("/kaggle/working/downloads")
CAFE_ZIP_URL = "https://zenodo.org/records/1478765/files/CaFE_48k.zip?download=1"
OPENSLR57_TAR_URL = "https://openslr.elda.org/resources/57/African_Accented_French.tar.gz"


def download_cafe() -> Path:
    # Le constructeur parcourt tous les WAV sous la racine : le dossier décompressé suffit.
    archive_path = DOWNLOAD_DIR / "cafe.zip"
    extract_dir = DOWNLOAD_DIR / "cafe"
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CAFE_ZIP_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
    archive_path.unlink()
    return extract_dir


def download_openslr57() -> Path:
    # La racine du jeu est le dossier African_Accented_French (speech/ et transcripts/).
    archive_path = DOWNLOAD_DIR / "openslr57.tar.gz"
    extract_dir = DOWNLOAD_DIR / "openslr57"
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(OPENSLR57_TAR_URL, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall(extract_dir, filter="data")
    archive_path.unlink()
    candidates = sorted(extract_dir.glob("**/African_Accented_French"))
    return candidates[0]


cafe_root = download_cafe()
print("CaFE :", cafe_root)
openslr57_root = download_openslr57()
print("OpenSLR 57 :", openslr57_root)

## b. Convention `<datasets_dir>/<id>`

Les manifestes donnent des chemins relatifs à `<datasets_dir>/<id du jeu>` (docs/datasets.md).
On crée donc `/kaggle/working/datasets/cafe` et `/kaggle/working/datasets/openslr57`, liens
symboliques vers les racines téléchargées.

In [ ]:
from echolens.core.data.manifest import dataset_root

datasets_dir = Path("/kaggle/working/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
for dataset_id, root in [("cafe", cafe_root), ("openslr57", openslr57_root)]:
    link = dataset_root(dataset_id, datasets_dir)
    if link.is_symlink():
        link.unlink()
    link.symlink_to(root)
    print(link, "→", root)

## c. Manifestes

Une ligne par fichier audio. CaFE garde son code d'émotion d'origine (P9), surprise `S`
comprise : elle n'est écartée qu'à l'évaluation (P2). OpenSLR 57 garde son découpage officiel,
son sous-corpus et sa transcription brute, non normalisée (P10, P12).

In [ ]:
from echolens.core.data.cafe import build_cafe_manifest
from echolens.core.data.manifest import save_manifest
from echolens.core.data.openslr57 import build_openslr57_manifest

cafe_rows = build_cafe_manifest(dataset_root("cafe", datasets_dir))
openslr57_rows = build_openslr57_manifest(dataset_root("openslr57", datasets_dir))
save_manifest(cafe_rows, Path("/kaggle/working/cafe.csv"))
save_manifest(openslr57_rows, Path("/kaggle/working/openslr57.csv"))
print("CaFE :", len(cafe_rows), "lignes")
print("OpenSLR 57 :", len(openslr57_rows), "lignes")

## d. Contrôles d'effectifs

Les chiffres attendus viennent de l'exploration des archives (936 WAV pour CaFE, 16 656 pour
OpenSLR 57). Au moindre écart, le notebook s'arrête avec la liste des différences. Toutes les
phrases du test officiel doivent avoir leur transcription, sans quoi le WER est impossible (P10).

In [ ]:
import statistics
from collections import Counter


def count_by(rows, field):
    counts = Counter()
    for row in rows:
        counts[getattr(row, field)] += 1
    return dict(sorted(counts.items()))


def speakers_by_gender(rows):
    # Un locuteur compte une fois, quel que soit son nombre de fichiers.
    gender_of_speaker = {}
    for row in rows:
        gender_of_speaker[row.speaker] = row.gender
    counts = Counter(gender_of_speaker.values())
    return dict(sorted(counts.items(), key=str))


def compare(label, observed, expected, problems):
    print(f"{label} : {observed}")
    if observed != expected:
        problems.append(f"{label} : obtenu {observed}, attendu {expected}")


def describe_durations(name, rows):
    durations = [row.duration_s for row in rows]
    print(
        f"{name} : durée min {min(durations):.3f} s, "
        f"médiane {statistics.median(durations):.3f} s, max {max(durations):.3f} s, "
        f"total {sum(durations) / 3600:.1f} h"
    )


problems = []

# CaFE : 12 acteurs (6 hommes, 6 femmes), 144 extraits par émotion, 72 pour le neutre.
cafe_expected_labels = {"N": 72}
for code_letter in ["C", "D", "J", "P", "S", "T"]:
    cafe_expected_labels[code_letter] = 144
compare("CaFE lignes", len(cafe_rows), 936, problems)
compare("CaFE acteurs", len(count_by(cafe_rows, "speaker")), 12, problems)
compare("CaFE acteurs par genre", speakers_by_gender(cafe_rows), {"F": 6, "M": 6}, problems)
compare(
    "CaFE extraits par émotion",
    count_by(cafe_rows, "label"),
    dict(sorted(cafe_expected_labels.items())),
    problems,
)

# OpenSLR 57 : effectifs par découpage officiel et par sous-corpus.
compare("OpenSLR 57 lignes", len(openslr57_rows), 16656, problems)
compare(
    "OpenSLR 57 par découpage",
    count_by(openslr57_rows, "official_split"),
    {"dev": 986, "devtest": 761, "test": 515, "train": 14394},
    problems,
)
compare(
    "OpenSLR 57 par sous-corpus",
    count_by(openslr57_rows, "subset"),
    {"ca16": 7273, "niger": 986, "yaounde": 8397},
    problems,
)

# Couverture des transcriptions, affichée pour chaque découpage et sous-corpus.
with_text = Counter()
total = Counter()
for row in openslr57_rows:
    group = (row.official_split, row.subset)
    total[group] += 1
    if row.transcript is not None:
        with_text[group] += 1
for group in sorted(total):
    print(f"Transcriptions {group} : {with_text[group]} / {total[group]}")
compare("Transcriptions du test officiel", with_text[("test", "ca16")], 515, problems)

describe_durations("CaFE", cafe_rows)
describe_durations("OpenSLR 57", openslr57_rows)

if problems:
    raise RuntimeError(
        "Effectifs inattendus, manifestes à ne pas utiliser :\n" + "\n".join(problems)
    )
print("Tous les effectifs sont conformes.")

## e. Aperçu

Quelques lignes de chaque manifeste, pour vérifier à l'œil les chemins et les transcriptions.

In [ ]:
import pandas as pd

print(pd.read_csv("/kaggle/working/cafe.csv").head(3).to_string())
openslr57_table = pd.read_csv("/kaggle/working/openslr57.csv")
for subset_name in ["ca16", "niger", "yaounde"]:
    sample = openslr57_table[openslr57_table["subset"] == subset_name].head(2)
    print(sample[["path", "official_split", "speaker", "gender", "transcript"]].to_string())

## Nettoyage

On supprime le clone du dépôt, les liens et les téléchargements : les sorties ne gardent que les
deux manifestes (I8).

In [ ]:
import shutil

shutil.rmtree(CLONE_DIR, ignore_errors=True)
shutil.rmtree(datasets_dir, ignore_errors=True)
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
remaining = [path.name for path in Path("/kaggle/working").iterdir()]
print(sorted(remaining))